# Water Quality Classification
Predicting water quality levels (VERDE / AMARILLO / ROJO) from chemical measurements using a Decision Tree classifier.

**Dataset:** Mexican groundwater monitoring data (2012–2024)  
**Target:** `SEMÁFORO` — a traffic-light quality indicator with three classes:
- 🟢 `VERDE` (0) — Good quality
- 🟡 `AMARILLO` (1) — Moderate concern
- 🔴 `ROJO` (2) — Poor quality

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from pathlib import Path

## Load Data

In [ ]:
df = pd.read_excel(Path("data") / "Calidad_del_Agua_Subterr_nea_p_2012-2024_15082025.xlsx")

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df["SEMÁFORO"].unique()

## Feature Selection

We keep only the 14 chemical measurement columns and the target variable `SEMAFORO`,
dropping metadata columns like site name, location, and administrative region.

In [ ]:
# Measured Chemical Values
features = ['ALC_mg/L',
            'CONDUCT_mS/cm',
            'SDT_mg/L',
            'FLUORUROS_mg/L',
            'DUR_mg/L',
            'COLI_FEC_NMP/100_mL',
            'N_NO3_mg/L',
            'AS_TOT_mg/L',
            'CD_TOT_mg/L',
            'CR_TOT_mg/L',
            'HG_TOT_mg/L',
            'PB_TOT_mg/L',
            'MN_TOT_mg/L',
            'FE_TOT_mg/L',
            'SEMÁFORO'
]

df = df[features]

## Data Cleaning

Most chemical columns were stored as `object` instead of `float` because some values
were recorded as detection limits (e.g. `<0.001`). We strip the `<` prefix and treat
those values as `0`, then convert all columns to `float`.

In [ ]:
df.info()

In [ ]:
# Columns to convert (all except SEMAFORO and CONDUCT which is already float)
cols_to_convert = [
    'ALC_mg/L', 'SDT_mg/L', 'FLUORUROS_mg/L', 'DUR_mg/L',
    'COLI_FEC_NMP/100_mL', 'N_NO3_mg/L', 'AS_TOT_mg/L',
    'CD_TOT_mg/L', 'CR_TOT_mg/L', 'HG_TOT_mg/L',
    'PB_TOT_mg/L', 'MN_TOT_mg/L', 'FE_TOT_mg/L'
]

def clean_numeric(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip()
    if val.startswith('<'):
        return 0  # or float(val[1:]) / 2 for midpoint
    try:
        return float(val)
    except ValueError:
        return np.nan  # catches any other unexpected strings

for col in cols_to_convert:
    df[col] = df[col].apply(clean_numeric).astype(float)

In [ ]:
df.info()

In [ ]:
df.head()

## Encode Target Variable

We map the string labels to integers so scikit-learn can work with them:

| Label | Encoded |
|---|---|
| VERDE | 0 |
| AMARILLO | 1 |
| ROJO | 2 |

In [ ]:
mapping = {'VERDE': 0, 'AMARILLO': 1, 'ROJO': 2}
df['SEMÁFORO'] = df['SEMÁFORO'].map(mapping)
df.head()

## Train / Test Split

We separate features (`X`) from the target (`y`) and split into 70% training and 30% test sets.

In [ ]:
# Entry variable
X = df.drop(columns=["SEMÁFORO"])

# Target variable
y = df["SEMÁFORO"]

X.shape, y.shape

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42
)

print("Train shapes:", X_train.shape, y_train.shape)
print("Test shapes: ", X_test.shape,  y_test.shape)

## Imputation

Some columns have missing values. We fill them with the **training set mean only** —
computing the mean on the full dataset before the split would leak information from
the test set into training, which would give overly optimistic results.

In [ ]:
df.isna().sum()

In [ ]:
# Fit on train only to prevent data leakage
imputation_features = [
    'ALC_mg/L',
    'CONDUCT_mS/cm',
    'SDT_mg/L',
    'FLUORUROS_mg/L',
    'DUR_mg/L',
    'COLI_FEC_NMP/100_mL',
    'N_NO3_mg/L',
    'AS_TOT_mg/L',
    'CD_TOT_mg/L',
    'CR_TOT_mg/L',
    'HG_TOT_mg/L',
    'PB_TOT_mg/L',
    'MN_TOT_mg/L',
    'FE_TOT_mg/L'
]

for col in imputation_features:
    train_mean = X_train[col].mean()
    X_train[col] = X_train[col].fillna(train_mean)
    X_test[col]  = X_test[col].fillna(train_mean)

print("Missing values after imputation:")
print(X_train.isna().sum().sum(), "in train —", X_test.isna().sum().sum(), "in test")

In [ ]:
X_train.iloc[1, :]

In [ ]:
# y = y.to_frame()   # convierte Serie a DataFrame
y_train.iloc[1]   # ahora sí funciona

## Model: Decision Tree

We train a Decision Tree classifier with `max_depth=6` to keep the model interpretable
and avoid overfitting to the training data.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Define the model
tree = DecisionTreeClassifier(max_depth=6, 
                              random_state=42,
                              class_weight='balanced')

# Train the model
tree.fit(X_train, y_train)

### Evaluation

We evaluate the model on both the training and test sets. A small gap between the two
accuracy scores indicates the model generalizes well without overfitting.

In [ ]:
from sklearn.metrics import accuracy_score

# Training predictions
y_train_pred = tree.predict(X_train)

# Test predictions
y_pred = tree.predict(X_test)

print("Train accuracy:", accuracy_score(y_train, y_train_pred))
print("Test accuracy: ", accuracy_score(y_test,  y_pred))

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred, target_names=['VERDE', 'AMARILLO', 'ROJO']))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_estimator(
    tree,
    X_test,
    y_test,
    display_labels=['VERDE', 'AMARILLO', 'ROJO'],
    cmap="Greens"
)
plt.title("Confusion Matrix - Decision Tree")
plt.show()

### Extract the model

In [ ]:
# Save (extract) the model
joblib.dump(tree, 'Decision-Tree_GroundWater-Model.joblib')

## Logistic Regression

In [ ]:
# StandardScaler — fit only on train to avoid data leakage
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000, random_state = 42)

log_reg.fit(X_train_scaled, y_train) 

In [ ]:
y_pred_log = log_reg.predict(X_test_scaled)

### Evaluation

We evaluate the model on both the training and test sets. A small gap between the two
accuracy scores indicates the model generalizes well without overfitting.

In [ ]:
# Training predictions
y_train_pred_log = log_reg.predict(X_train_scaled)

# Test predictions
y_pred_log = log_reg.predict(X_test_scaled)

print("Train accuracy:", accuracy_score(y_train, y_train_pred_log))
print("Test accuracy: ", accuracy_score(y_test,  y_pred_log))

In [ ]:
print(classification_report(y_test, y_pred_log, target_names=['VERDE', 'AMARILLO', 'ROJO']))

In [ ]:
ConfusionMatrixDisplay.from_estimator(
    log_reg,
    X_test_scaled,
    y_test,
    display_labels=['VERDE', 'AMARILLO', 'ROJO'],
    cmap="Reds"
)
plt.title("Confusion Matrix - Logistic Regression")
plt.show()

## Supported Vector Machine

In [ ]:
from sklearn.svm import SVC

svm = SVC(
    C            = 1000,      # Was 100 — more freedom to fit the data
    gamma        = 0.01,      # Was 0.1 — less aggressive kernel, better generalization
    kernel       = 'rbf',
    class_weight = 'balanced',
    random_state = 42,
)

# Training model
svm.fit(X_train_scaled, y_train)

In [ ]:
y_pred_svm = svm.predict(X_test_scaled)

### Evaluation
We evaluate the model on both the training and test sets. A small gap between the two accuracy scores indicates the model generalizes well without overfitting.

In [ ]:
# Training predictions
y_train_pred_svm = svm.predict(X_train_scaled)

# Test predictions
y_pred_svm = svm.predict(X_test_scaled)

print("Train accuracy:", accuracy_score(y_train, y_train_pred_svm))
print("Test accuracy: ", accuracy_score(y_test,  y_pred_svm))

In [ ]:
print(classification_report(y_test, y_pred_svm, target_names=['VERDE', 'AMARILLO', 'ROJO']))

In [ ]:
ConfusionMatrixDisplay.from_estimator(
    svm,
    X_test_scaled,
    y_test,
    display_labels=['VERDE', 'AMARILLO', 'ROJO'],
    cmap="YlOrBr"
)
plt.title("Confusion Matrix - Supported Vector Machine (SVM)")
plt.show()

## Neural Network (PyTorch)

In [ ]:
import torch

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Using device:", device)

X_train_t = torch.from_numpy(X_train_scaled.astype(np.float32)).to(device)
X_test_t  = torch.from_numpy(X_test_scaled.astype(np.float32)).to(device)
y_train_t = torch.from_numpy(y_train.values.copy()).long().to(device)
y_test_t  = torch.from_numpy(y_test.values.copy()).long().to(device)

NUM_FEATURES = X_train_t.shape[1]
NUM_CLASSES  = len(torch.unique(y_train_t))
print(f"Features: {NUM_FEATURES} | Classes: {NUM_CLASSES}")

In [ ]:
X_train_t[:5], y_train_t[:5]

In [ ]:
# Build a multi-class classification model
import torch.nn as nn

class WaterModel(nn.Module):
    def __init__(self, input_features, output_features, hidden_units):
        """ Initialize multi_class classification model.

        Args: 
            input_features (int): Number of input features to the model
            output_features (int): Number of output features (number of output classes)
            hidden_units (int): Number of hidden units between layers, default 8

        Returns:

        Example:
        """
        super().__init__()
        self.linear_layer_stack = nn.Sequential(
            nn.Linear(input_features, hidden_units),
            nn.BatchNorm1d(hidden_units),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden_units, hidden_units),
            nn.BatchNorm1d(hidden_units),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden_units, hidden_units // 2),
            nn.BatchNorm1d(hidden_units // 2),
            nn.ReLU(),

            nn.Linear(hidden_units // 2, output_features)
        )
    
    def forward(self, x):
        return self.linear_layer_stack(x)

# Create an instance of WaterModel
torch.manual_seed(42)
model_0 = WaterModel(input_features = NUM_FEATURES,
                     output_features = NUM_CLASSES,
                     hidden_units=64).to(device)
print(model_0)

In [ ]:
model_0.state_dict()

In [ ]:
X_train_t.shape, y_train_t.shape

In [ ]:
torch.unique(y_train_t)

### Create a loss function and an optimizer for multi_class classification model

The model has unbalanced samples, weight parameter has to change for unbalanced datasets

In [ ]:
class_counts  = torch.bincount(y_train_t)
class_weights = 1.0 / class_counts.float()
class_weights = class_weights / class_weights.sum() * NUM_CLASSES
class_weights = class_weights.to(device)
print("Class weights:", class_weights)

loss_fn   = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model_0.parameters(), lr=1e-3, weight_decay=1e-4)

### Creating a training loop and testing loop for a multi-class PyTorch model

Logits (raw output of the model) -> probabilites (`torch.softmax`) -> pred labels (take the argmax of the prediction probabilites)

In [ ]:
def accuracy_fn(y_true, y_pred):
    return (torch.eq(y_true, y_pred).sum().item() / len(y_pred)) * 100

In [ ]:
# Fit the multi-class model to data
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# Set number of epochs
epochs = 800

# The data is already on the target device


# Loop through data
for epoch in range(epochs):
    ### Training
    model_0.train()

    y_logits = model_0(X_train_t)
    y_pred = torch.softmax(y_logits, dim=1).argmax(dim=1)

    loss = loss_fn(y_logits, y_train_t)
    acc = accuracy_fn(y_true=y_train_t,
                      y_pred=y_pred)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    ### Testing
    model_0.eval()
    with torch.inference_mode():
        test_logits = model_0(X_test_t)
        test_preds = torch.softmax(test_logits, dim=1).argmax(dim=1)

        test_loss = loss_fn(test_logits, y_test_t)
        test_acc = accuracy_fn(y_true=y_test_t,
                               y_pred=test_preds)

    # Print out what's happening
    if epoch % 100 == 0:
        print(f"Epoch: {epoch} | Train Loss: {loss:.4f}, Acc: {acc:.2f}% | Test loss: {test_loss:.4f}, Test acc: {test_acc:.4f}")

### Evaluation

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import numpy as np

# Get predicted probabilities on test set
model_0.eval()
with torch.inference_mode():
    test_logits = model_0(X_test_t)
    test_probs = torch.softmax(test_logits, dim=1).cpu().numpy()

# Get true labels
y_true = y_test_t.cpu().numpy()

# Determine number of classes
n_classes = test_probs.shape[1]

# Binarize the labels for One-vs-Rest ROC
y_true_bin = label_binarize(y_true, classes=list(range(n_classes)))

# --- Compute ROC curve and AUC for each class ---
fpr, tpr, roc_auc = {}, {}, {}

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], test_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# --- Compute Macro-average ROC ---
all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(n_classes):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
mean_tpr /= n_classes

fpr["macro"], tpr["macro"] = all_fpr, mean_tpr
roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

# --- Plot ---
fig, ax = plt.subplots(figsize=(8, 6))

colors = plt.cm.Set1(np.linspace(0, 0.8, n_classes))

for i, color in zip(range(n_classes), colors):
    ax.plot(fpr[i], tpr[i], color=color, lw=1.8,
            label=f"Class {i} (AUC = {roc_auc[i]:.2f})")

ax.plot(fpr["macro"], tpr["macro"], color="navy",
        linestyle="--", lw=2.5,
        label=f"Macro-avg (AUC = {roc_auc['macro']:.2f})")

ax.plot([0, 1], [0, 1], "k--", lw=1.2, label="Random Classifier")

ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel("False Positive Rate", fontsize=13)
ax.set_ylabel("True Positive Rate", fontsize=13)
ax.set_title("Multi-Class ROC Curve (One-vs-Rest)", fontsize=15)
ax.legend(loc="lower right", fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

# --- Get predictions ---
model_0.eval()
with torch.inference_mode():
    # Train predictions
    train_logits = model_0(X_train_t)
    train_preds = torch.softmax(train_logits, dim=1).argmax(dim=1).cpu().numpy()

    # Test predictions
    test_logits = model_0(X_test_t)
    test_preds = torch.softmax(test_logits, dim=1).argmax(dim=1).cpu().numpy()

# --- True labels ---
y_train_np = y_train_t.cpu().numpy()
y_test_np  = y_test_t.cpu().numpy()

# --- Accuracies ---
train_acc = accuracy_score(y_train_np, train_preds)
test_acc  = accuracy_score(y_test_np,  test_preds)

# --- Class names (adjust order to match your label encoding) ---
class_names = ["VERDE", "AMARILLO", "ROJO"]  # ← change if needed

# --- Report ---
report = classification_report(
    y_test_np,
    test_preds,
    target_names=class_names
)

print("=" * 55)
print("   Neural Network (PyTorch)")
print("=" * 55)
print(f"   Train accuracy : {train_acc:.4f}")
print(f"   Test  accuracy : {test_acc:.4f}")
print()
print(report)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# --- Compute confusion matrix ---
cm = confusion_matrix(y_test_np, test_preds)

# --- Plot ---
fig, ax = plt.subplots(figsize=(7, 6))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names        # ["VERDE", "AMARILLO", "ROJO"]
)

disp.plot(
    cmap="Blues",
    colorbar=False,
    ax=ax,
    values_format="d"                 # show integers, not scientific notation
)

ax.set_title("Confusion Matrix — Neural Network (PyTorch)", fontsize=14, pad=15)
ax.set_xlabel("Predicted Label", fontsize=12)
ax.set_ylabel("True Label", fontsize=12)
ax.tick_params(axis='both', labelsize=11)

plt.tight_layout()
plt.show()

### Extract the Model

In [ ]:
# save it directly to your desired folder
torch.save(model_0.state_dict(), '/Users/alberto/Desktop/ML-Water/model_weights.pth')

In [ ]:
torch.save(model_0, 'NN_GroundWater-Model.pth')

In [ ]:
joblib.dump(scaler, "scaler.pkl")